In [ ]:
from pathlib import Path
import calendar
import json
import random
import re
import time
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rasterio

from rasterio.features import geometry_mask
from rasterio.mask import mask
from shapely.geometry import mapping


# -------------------------------------------------
# SETTINGS
# -------------------------------------------------
sentinel2_grid_kml = "S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml"
kml_layer = "Features"

tile_name = "22MFA"
start_year = 2019
end_year = 2021

catalog_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
collection_name = "sentinel-2-l2a"

# Evaluate all scenes first, then filter by coverage, then keep top 10 by cloud
shortlist_n = None
TOP_N_AFTER_COVERAGE_FILTER = 10
MIN_COVERAGE_FRACTION = 0.90

# SCL classes considered usable / clear
CLEAR_SCL = np.array([4, 5, 6], dtype=np.int16)

# cloud-like classes for reporting
CLOUD_SCL = np.array([1,2,3,7,8, 9,11], dtype=np.int16)
CIRRUS_SCL = np.array([10], dtype=np.int16)

sign_retries = 5
sign_base_sleep = 1.5

# Raw assets to save separately
ASSETS_TO_SAVE = [
    "B01", "B02", "B03", "B04",
    "B05", "B06", "B07", "B08",
    "B8A", "B09", "B10", "B11", "B12",
    "SCL",
]

output_dir = Path("/media/sara/Extreme Pro") / f"pc_s2_best_allbands_{tile_name}_{start_year}_{end_year}_monthly"
output_dir.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------
# MONTHLY PERIODS
# -------------------------------------------------
def build_month_periods(start_year, end_year):
    periods = {}
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            last_day = calendar.monthrange(year, month)[1]
            month_label = f"{year}_{month:02d}"
            periods[month_label] = f"{year}-{month:02d}-01/{year}-{month:02d}-{last_day:02d}"
    return periods


periods = build_month_periods(start_year, end_year)


# -------------------------------------------------
# HELPERS
# -------------------------------------------------
def union_all_compat(gdf):
    try:
        return gdf.union_all()
    except Exception:
        return gdf.geometry.unary_union


def pick_tile_id_column(gdf):
    candidates = ["Name", "name", "tile", "tile_id", "TILE_ID", "mgrs", "MGRS"]
    for c in candidates:
        if c in gdf.columns:
            return c
    raise ValueError(f"Could not find tile ID column. Available columns: {list(gdf.columns)}")


def get_cloud_cover(item):
    value = item.properties.get("eo:cloud_cover")
    if value is None:
        return np.inf
    try:
        return float(value)
    except Exception:
        return np.inf


def get_mgrs_tile(item):
    for key in ["s2:mgrs_tile", "mgrs:tile", "tile"]:
        value = item.properties.get(key)
        if value is not None:
            return str(value).upper().strip()

    m = re.search(r"_T?(\d{2}[A-Z]{3})_", str(item.id).upper())
    if m:
        return m.group(1)

    return None


def choose_asset_key(item, candidates):
    key_map = {str(k).lower(): k for k in item.assets.keys()}
    for cand in candidates:
        if str(cand).lower() in key_map:
            return key_map[str(cand).lower()]
    return None


def get_asset_key(item, logical_name):
    key = choose_asset_key(item, [logical_name])
    if key is not None:
        return key

    aliases = {
        "SCL": ["scene_classification", "scl", "SCL_20m"],
        "B02": ["blue"],
        "B03": ["green"],
        "B04": ["red"],
    }

    for cand in aliases.get(logical_name, []):
        key = choose_asset_key(item, [cand])
        if key is not None:
            return key

    return None


def ordered_available_assets(item):
    found = []
    for logical_name in ASSETS_TO_SAVE:
        key = get_asset_key(item, logical_name)
        if key is not None:
            found.append((logical_name, key))
    if not found:
        raise ValueError(f"No requested assets found for {item.id}. Assets: {list(item.assets.keys())}")
    return found


def as_float_array(x):
    if np.ma.isMaskedArray(x):
        return x.astype("float32", copy=False).filled(np.nan)
    return np.asarray(x, dtype="float32")


def sign_href(asset, retries=5, base_sleep=1.5):
    last_err = None

    for attempt in range(retries):
        try:
            return planetary_computer.sign(asset).href
        except Exception as e:
            last_err = e
            if attempt == retries - 1:
                break

            sleep_s = base_sleep * (2 ** attempt) + random.uniform(0.2, 1.0)
            print(f"    signing retry {attempt + 1}/{retries - 1} after error: {e}")
            time.sleep(sleep_s)

    raise last_err


def clip_asset_raw(asset_href, area_gdf):
    with rasterio.open(asset_href) as src:
        area_proj = area_gdf.to_crs(src.crs)
        area_union = union_all_compat(area_proj)
        geom = [mapping(area_union)]

        data_masked, transform = mask(
            src,
            geom,
            crop=True,
            filled=False,
        )

        inside_geom = geometry_mask(
            geom,
            out_shape=data_masked.shape[1:],
            transform=transform,
            invert=True,
        )

        return {
            "data": data_masked[0],
            "transform": transform,
            "crs": src.crs,
            "inside_geom": inside_geom,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
        }


def save_raw_single_band_tif(path, clipped_info, band_description):
    data_ma = clipped_info["data"]
    transform = clipped_info["transform"]
    crs = clipped_info["crs"]
    src_dtype = np.dtype(clipped_info["dtype"])
    src_nodata = clipped_info["nodata"]

    if src_nodata is None:
        if np.issubdtype(src_dtype, np.integer):
            fill_value = 0
        else:
            fill_value = np.nan
    else:
        fill_value = src_nodata

    filled = data_ma.filled(fill_value).astype(src_dtype, copy=False)

    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=filled.shape[0],
        width=filled.shape[1],
        count=1,
        dtype=src_dtype.name,
        crs=crs,
        transform=transform,
        nodata=fill_value,
        compress="deflate",
        tiled=True,
    ) as dst:
        dst.write(filled, 1)
        dst.set_band_description(1, band_description)


def compute_candidate_stats(item, tile_gdf):
    scl_key = get_asset_key(item, "SCL")
    if scl_key is None:
        raise ValueError(f"No SCL asset found for {item.id}")

    scl_href = sign_href(
        item.assets[scl_key],
        retries=sign_retries,
        base_sleep=sign_base_sleep,
    )

    scl_info = clip_asset_raw(scl_href, tile_gdf)

    scl = as_float_array(scl_info["data"])
    inside = scl_info["inside_geom"]

    inside_count = int(inside.sum())
    if inside_count == 0:
        raise ValueError(f"Inside geometry is empty for {item.id}")

    finite = inside & np.isfinite(scl)

    scl_int = np.full(scl.shape, -9999, dtype=np.int16)
    scl_int[finite] = np.rint(scl[finite]).astype(np.int16)

    nodata_mask = inside & ((~finite) | (scl_int == 0))
    covered_mask = inside & finite & (scl_int != 0)
    clear_mask = covered_mask & np.isin(scl_int, CLEAR_SCL)
    cloud_mask = covered_mask & np.isin(scl_int, CLOUD_SCL)
    cirrus_mask = covered_mask & np.isin(scl_int, CIRRUS_SCL)

    coverage_fraction = float(covered_mask.sum() / inside_count)
    usable_fraction = float(clear_mask.sum() / inside_count)
    nodata_fraction = float(nodata_mask.sum() / inside_count)
    cloud_fraction = float(cloud_mask.sum() / inside_count)
    cirrus_fraction = float(cirrus_mask.sum() / inside_count)

    return {
        "item": item,
        "item_id": item.id,
        "datetime": pd.to_datetime(item.datetime),
        "mgrs_tile": get_mgrs_tile(item),
        "eo_cloud_cover": get_cloud_cover(item),
        "coverage_fraction": coverage_fraction,
        "usable_fraction": usable_fraction,
        "cloud_fraction": cloud_fraction,
        "cirrus_fraction": cirrus_fraction,
        "nodata_fraction": nodata_fraction,
    }


def export_best_data(month_label, winner_row, tile_gdf):
    item = winner_row["item"]
    date_str = str(pd.to_datetime(winner_row["datetime"]).date())

    month_dir = output_dir / month_label
    month_dir.mkdir(parents=True, exist_ok=True)

    prefix = month_dir / f"{tile_name}_{month_label}_{date_str}_{item.id}"
    item_json = f"{prefix}_item.json"

    saved_files = {}

    assets_found = ordered_available_assets(item)

    print("\nSaving raw clipped assets:")
    for logical_name, asset_key in assets_found:
        try:
            href = sign_href(
                item.assets[asset_key],
                retries=sign_retries,
                base_sleep=sign_base_sleep,
            )
            clipped = clip_asset_raw(href, tile_gdf)

            out_tif = f"{prefix}_{logical_name}.tif"
            save_raw_single_band_tif(out_tif, clipped, logical_name)

            saved_files[logical_name] = str(out_tif)
            print(f"  saved {logical_name}: {out_tif}")

        except Exception as e:
            print(f"  could not save {logical_name}: {e}")

    with open(item_json, "w", encoding="utf-8") as f:
        json.dump(item.to_dict(), f, indent=2)

    result = {
        "month_label": month_label,
        "time_range": periods[month_label],
        "item_id": item.id,
        "date": date_str,
        "coverage_fraction": float(winner_row["coverage_fraction"]),
        "usable_fraction": float(winner_row["usable_fraction"]),
        "cloud_fraction": float(winner_row["cloud_fraction"]),
        "cirrus_fraction": float(winner_row["cirrus_fraction"]),
        "nodata_fraction": float(winner_row["nodata_fraction"]),
        "eo_cloud_cover": float(winner_row["eo_cloud_cover"]),
        "item_json": str(item_json),
    }

    for logical_name in ASSETS_TO_SAVE:
        result[f"{logical_name}_tif"] = saved_files.get(logical_name)

    return result


def run_month(catalog, month_label, time_range, tile_gdf, tile_geom):
    print("\n" + "=" * 80)
    print(f"MONTH: {month_label} | {time_range}")
    print("=" * 80)

    search = catalog.search(
        collections=[collection_name],
        intersects=tile_geom.__geo_interface__,
        datetime=time_range,
    )

    items = list(search.items())
    print(f"Found {len(items)} raw items")

    if not items:
        warnings.warn(f"No Sentinel-2 items found for {month_label}")
        return None

    exact_items = [item for item in items if get_mgrs_tile(item) == tile_name]
    if exact_items:
        items = exact_items
        print(f"After exact tile filter: {len(items)} items")
    else:
        print("No exact MGRS tile property match found; using spatial hits.")

    meta_df = pd.DataFrame({
        "item": items,
        "item_id": [item.id for item in items],
        "datetime": [pd.to_datetime(item.datetime) for item in items],
        "eo_cloud_cover": [get_cloud_cover(item) for item in items],
        "mgrs_tile": [get_mgrs_tile(item) for item in items],
    }).sort_values(
        ["eo_cloud_cover", "datetime"],
        ascending=[True, True],
    ).reset_index(drop=True)

    if shortlist_n is not None and len(meta_df) > shortlist_n:
        meta_df = meta_df.head(shortlist_n)
        print(f"Evaluating only top {len(meta_df)} metadata-ranked candidates")

    results = []
    for i, row in meta_df.iterrows():
        item = row["item"]
        print(f"Evaluating {i + 1}/{len(meta_df)}: {item.id}")

        try:
            stats = compute_candidate_stats(item, tile_gdf)
            results.append(stats)
            print(
                f"  date={stats['datetime'].date()} | "
                f"eo_cloud={stats['eo_cloud_cover']:.2f} | "
                f"coverage={stats['coverage_fraction']:.3f} | "
                f"nodata={stats['nodata_fraction']:.3f} | "
                f"usable={stats['usable_fraction']:.3f} | "
                f"cloud={stats['cloud_fraction']:.3f} | "
                f"cirrus={stats['cirrus_fraction']:.3f}"
            )
        except Exception as e:
            print(f"  skipped because of error: {e}")

    if not results:
        warnings.warn(f"No candidate could be evaluated for {month_label}")
        return None

    summary_df = pd.DataFrame(results)

    # 1) keep only scenes with enough tile coverage
    summary_df = summary_df[
        summary_df["coverage_fraction"] >= MIN_COVERAGE_FRACTION
    ].copy()

    if summary_df.empty:
        warnings.warn(f"No candidate passed coverage >= {MIN_COVERAGE_FRACTION:.2f} for {month_label}")
        return None

    # 2) sort by lowest cloud first
    summary_df = summary_df.sort_values(
        [
            "eo_cloud_cover",
            "nodata_fraction",
            "usable_fraction",
            "cloud_fraction",
            "cirrus_fraction",
            "datetime",
        ],
        ascending=[True, True, False, True, True, True],
    ).reset_index(drop=True)

    # 3) keep the 10 best after coverage filter
    summary_df = summary_df.head(TOP_N_AFTER_COVERAGE_FILTER).reset_index(drop=True)

    print("\nTop candidates after coverage filter:")
    print(
        summary_df[
            [
                "datetime",
                "eo_cloud_cover",
                "coverage_fraction",
                "nodata_fraction",
                "usable_fraction",
                "cloud_fraction",
                "cirrus_fraction",
                "mgrs_tile",
                "item_id",
            ]
        ]
    )

    # 4) choose the first one
    winner = summary_df.iloc[0].to_dict()

    print("\nSelected winner:")
    print("  ID:", winner["item_id"])
    print("  Date:", pd.to_datetime(winner["datetime"]).date())
    print("  EO cloud cover:", f"{winner['eo_cloud_cover']:.2f}")
    print("  Coverage:", f"{winner['coverage_fraction']:.3f}")
    print("  Nodata:", f"{winner['nodata_fraction']:.3f}")
    print("  Usable:", f"{winner['usable_fraction']:.3f}")
    print("  Cloud:", f"{winner['cloud_fraction']:.3f}")
    print("  Cirrus:", f"{winner['cirrus_fraction']:.3f}")

    exported = export_best_data(month_label, winner, tile_gdf)

    ranking_csv = output_dir / month_label / f"{tile_name}_{month_label}_candidate_ranking.csv"
    summary_df.drop(columns=["item"]).to_csv(ranking_csv, index=False)
    exported["candidate_ranking_csv"] = str(ranking_csv)

    return exported


# -------------------------------------------------
# LOAD TILE
# -------------------------------------------------
tiles_gdf = gpd.read_file(sentinel2_grid_kml, layer=kml_layer) if kml_layer else gpd.read_file(sentinel2_grid_kml)

if tiles_gdf.crs is None:
    raise ValueError("Tile grid has no CRS")

tiles_gdf = tiles_gdf.to_crs(epsg=4326)
tile_id_col = pick_tile_id_column(tiles_gdf)
tiles_gdf[tile_id_col] = tiles_gdf[tile_id_col].astype(str).str.strip().str.upper()

tile_gdf = tiles_gdf[tiles_gdf[tile_id_col] == tile_name].copy()
if tile_gdf.empty:
    raise ValueError(f"Tile {tile_name} not found in {sentinel2_grid_kml}")

tile_geom = union_all_compat(tile_gdf)

print(f"Loaded tile: {tile_name}")
print("Tile bounds:", tile_gdf.total_bounds)
print(f"Saving outputs to: {output_dir.resolve()}")


# -------------------------------------------------
# RUN ALL MONTHS
# -------------------------------------------------
catalog = pystac_client.Client.open(catalog_url)

exports = []
failed_months = []

for month_label, time_range in periods.items():
    result = run_month(catalog, month_label, time_range, tile_gdf, tile_geom)
    if result is not None:
        exports.append(result)
    else:
        failed_months.append({
            "month_label": month_label,
            "time_range": time_range,
            "reason": "no_candidate_after_coverage_filter_or_no_data",
        })

if not exports:
    raise ValueError("No outputs were produced for any month.")

exports_df = pd.DataFrame(exports).sort_values("month_label").reset_index(drop=True)

summary_csv = output_dir / f"{tile_name}_{start_year}_{end_year}_best_allbands_monthly_summary.csv"
exports_df.to_csv(summary_csv, index=False)

failed_csv = output_dir / f"{tile_name}_{start_year}_{end_year}_failed_months.csv"
pd.DataFrame(failed_months).to_csv(failed_csv, index=False)

print("\nDone.")
print(f"Summary CSV: {summary_csv}")
print(f"Failed months CSV: {failed_csv}")

cols_to_show = [
    "month_label",
    "date",
    "eo_cloud_cover",
    "coverage_fraction",
    "nodata_fraction",
    "usable_fraction",
    "cloud_fraction",
    "cirrus_fraction",
    "item_id",
]

for asset_name in ASSETS_TO_SAVE:
    col = f"{asset_name}_tif"
    if col in exports_df.columns:
        cols_to_show.append(col)

print(exports_df[cols_to_show])